# 01 — Quickstart: Load a model and generate text

The three-step workflow for every inference task:

```
ModelConfig  →  LLM.load()  →  LLM.generate_text()
```

> **Colab users:** Run the setup cell, restart the runtime, then continue.

In [ ]:
import os, sys, subprocess

def _is_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

IN_COLAB = _is_colab()

if IN_COLAB:
    repo = 'MyLLM'
    # Clone repo if not present
    if not os.path.exists(repo):
        print('Cloning MyLLM...')
        r = subprocess.run(
            ['git', 'clone', 'https://github.com/silvaxxx1/MyLLM.git', repo],
            capture_output=True, text=True
        )
        if r.returncode != 0:
            print('Clone FAILED:\n', r.stderr)
            raise RuntimeError('Clone failed')
    # Always uninstall stale version, then reinstall from local clone
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', 'myllm', '-y'],
                   capture_output=True)
    print('Installing myllm from local clone...')
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', f'./{repo}'],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        print('Install FAILED. Error:')
        print(r.stdout[-3000:])
        print(r.stderr[-3000:])
    else:
        print('Done! Now go to Runtime → Restart runtime, then continue.')
else:
    # Local dev — editable install from repo root
    root = os.path.abspath(os.path.join(os.getcwd(), '..'))
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-e', root],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        print('Install FAILED:', r.stderr[-2000:])
    else:
        print('Local install ready.')


## 1. Inspect a ModelConfig

In [ ]:
import torch
from myllm import ModelConfig

cfg = ModelConfig.from_name('gpt2-small')

print(f'Model      : {cfg.name}')
print(f'Layers     : {cfg.n_layer}')
print(f'Heads      : {cfg.n_head}')
print(f'Embedding  : {cfg.n_embd}')
print(f'Vocab size : {cfg.vocab_size}')
print(f'Block size : {cfg.block_size}')

mem = cfg.estimate_memory(batch_size=1, dtype=torch.float32)
print(f'\nMemory estimate (fp32):')
print(f'  Parameters : {mem["n_parameters"]/1e6:.1f}M  ({mem["parameters_gb"]:.2f} GB)')
print(f'  Activations: {mem["activations_gb"]:.3f} GB')
print(f'  Total      : {mem["total_gb"]:.2f} GB')

## 2. Load the model

`LLM.load()` downloads weights from HuggingFace on first run, caches them locally, maps them to the GPT architecture, and puts the model in eval mode.

> **On Colab:** weights are cached in `/content/models/` for the session.

In [ ]:
import torch
from myllm import LLM, ModelConfig

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

cfg = ModelConfig.from_name('gpt2-small')
llm = LLM(config=cfg, device=device)
llm.load('gpt2-small', 'gpt2')

## 3. Tokenizer

In [ ]:
from transformers import GPT2Tokenizer as HFTokenizer

tokenizer = HFTokenizer.from_pretrained('gpt2')
tokenizer.pad_token_id = tokenizer.eos_token_id

## 4. `generate_text()` — string in, string out

In [ ]:
from myllm import GenerationConfig

gen_cfg = GenerationConfig(
    max_length=60,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    apply_top_k_sampling=True,
    apply_top_p_sampling=True,
    do_sample=True,
    use_kv_cache=True,
)

result = llm.generate_text(
    prompt='The future of AI is',
    tokenizer=tokenizer,
    generation_config=gen_cfg,
)

print('Generated text:')
print('-' * 60)
print(result['text'])

## 5. `generate()` — token tensor in, token tensor out

In [ ]:
import torch
from myllm import GenerationConfig

prompt = 'In a world where machines think,'
input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)
print('Prompt shape:', input_ids.shape)  # (1, seq_len)

gen_cfg = GenerationConfig(
    max_length=40,
    temperature=1.0,
    top_k=50,
    apply_top_k_sampling=True,
    do_sample=True,
    use_kv_cache=True,
    return_logprobs=True,
)

output = llm.generate(input_ids, gen_cfg)

print('Output tokens shape :', output['tokens'].shape)
print('Log-probs shape     :', output['logprobs'].shape)
print()
print(tokenizer.decode(output['tokens'][0], skip_special_tokens=True))

## 6. `generate_batch()` — multiple prompts at once

In [ ]:
from myllm import GenerationConfig

prompts = [
    'Once upon a time',
    'The capital of France is',
    'To train a neural network you need',
]

gen_cfg = GenerationConfig(
    max_length=30,
    temperature=0.9,
    top_k=40,
    apply_top_k_sampling=True,
    do_sample=True,
    use_kv_cache=False,  # KV cache not supported with padded batches
)

results = llm.generate_batch(prompts, tokenizer, gen_cfg)

for i, res in enumerate(results):
    print(f'[{i}] {res["text"]}')
    print()

## 7. List cached models

In [ ]:
print('Models cached on disk:')
for m in llm.list_models():
    print(' ', m)